# Data Preprocessing 
Load raw data, visualize, map labels, chunk windows, and normalize.

In [43]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/AI engineering/COS40007-Project/backend')

# Importing libraries

In [44]:
# ====================== SETUP & IMPORTS ======================
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

from utils.seed import set_seed
from utils.device import get_device
from data.preprocessing import create_windows, clean_features
from data.motion_dataset import MotionDataset

# Config
seed = 42
device = get_device("auto")
set_seed(seed)

sensor_types = ["Segment Velocity", "Segment Acceleration"]
window_size = 90          # Increased a bit for better rolling stats
stride = 45
batch_size = 32
num_workers = 0
val_split = 0.2
n_folds = 10

print(f"[setup] Using device: {device}")

[setup] Using device: cpu


# Load data 

In [45]:
print("[phase=data] loading P1 + P2 datasets")
DATA_DIR = PROJECT_ROOT / "output_data"   # adjust if needed

p1_dfs = [pd.read_csv(DATA_DIR / "P1_boning.csv"), pd.read_csv(DATA_DIR / "P1_slicing.csv")]
p1_raw_df = pd.concat(p1_dfs, ignore_index=True)

p2_dfs = [pd.read_csv(DATA_DIR / "P2_boning.csv"), pd.read_csv(DATA_DIR / "P2_slicing.csv")]
p2_raw_df = pd.concat(p2_dfs, ignore_index=True)

p1_raw_df = p1_raw_df[p1_raw_df["sensor_type"].isin(sensor_types)]
p2_raw_df = p2_raw_df[p2_raw_df["sensor_type"].isin(sensor_types)]

def merge_velocity_and_acceleration(df):
    vel_df = df[df["sensor_type"] == "Segment Velocity"].copy()
    acc_df = df[df["sensor_type"] == "Segment Acceleration"].copy()

    base_feature_cols = get_feature_columns(vel_df)   # from your utils

    id_cols = ["video_id", "Frame", "Label", "person_id", "activity_type", "knife_sharpness_score"]

    vel_df = vel_df[id_cols + base_feature_cols].rename(
        columns={c: f"{c}_vel" for c in base_feature_cols}
    )
    acc_df = acc_df[["video_id", "Frame"] + base_feature_cols].rename(
        columns={c: f"{c}_acc" for c in base_feature_cols}
    )

    merged = vel_df.merge(acc_df, on=["video_id", "Frame"], how="inner")
    return merged, base_feature_cols

p1_df, base_feature_cols = merge_velocity_and_acceleration(p1_raw_df)
p2_df, _ = merge_velocity_and_acceleration(p2_raw_df)

print(f"[phase=data] p1_rows={len(p1_df)} p2_rows={len(p2_df)}")

[phase=data] loading P1 + P2 datasets


D:\Temp\ipykernel_5972\1149715951.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  p1_dfs = [pd.read_csv(DATA_DIR / "P1_boning.csv"), pd.read_csv(DATA_DIR / "P1_slicing.csv")]


[phase=data] p1_rows=61844 p2_rows=104214


# Visualize Data (Optional)

In [46]:
def map_sharpness(score: int) -> str:
    if score >= 85:
        return "sharp"
    elif score >= 70:
        return "medium"
    else:
        return "blunt"

p1_df["sharpness_class"] = p1_df["knife_sharpness_score"].apply(map_sharpness)
p2_df["sharpness_class"] = p2_df["knife_sharpness_score"].apply(map_sharpness)

target_col = "sharpness_class"

print("[phase=data] Sharpness class distribution (P1):")
print(p1_df[target_col].value_counts())
print("\n[phase=data] Sharpness class distribution (P2):")
print(p2_df[target_col].value_counts())

[phase=data] Sharpness class distribution (P1):
sharpness_class
blunt    40114
sharp    21730
Name: count, dtype: int64

[phase=data] Sharpness class distribution (P2):
sharpness_class
blunt     50440
sharp     37521
medium    16253
Name: count, dtype: int64


# Process Data: Split Train/Test/Val

In [47]:
print("[phase=features] Adding 5 new feature groups...")

def mag(df, cols):
    return np.sqrt((df[cols]**2).sum(axis=1))

acc_cols = [c for c in p1_df.columns if str(c).endswith("_acc")]
vel_cols = [c for c in p1_df.columns if str(c).endswith("_vel")]

# 1. Jerk
jerk_cols = []
for col in acc_cols:
    jerk_name = str(col).replace("_acc", "_jerk")
    p1_df[jerk_name] = p1_df.groupby("video_id")[col].diff().fillna(0)
    p2_df[jerk_name] = p2_df.groupby("video_id")[col].diff().fillna(0)
    jerk_cols.append(jerk_name)

# 2. Acc/Vel ratio
for v, a in zip(vel_cols, acc_cols):
    ratio_name = str(v).replace("_vel", "_acc_vel_ratio")
    p1_df[ratio_name] = p1_df[a] / (p1_df[v].abs() + 1e-8)
    p2_df[ratio_name] = p2_df[a] / (p2_df[v].abs() + 1e-8)

# 3. Smoothness index
for j, a in zip(jerk_cols, acc_cols):
    smooth_name = str(j).replace("_jerk", "_smoothness")
    p1_df[smooth_name] = p1_df[j].abs() / (p1_df[a].abs() + 1e-8)
    p2_df[smooth_name] = p2_df[j].abs() / (p2_df[a].abs() + 1e-8)

# 4. Hand energy ratio
hand_cols = [c for c in vel_cols if any(x in str(c) for x in ["Hand", "Forearm"])]
body_cols = [c for c in vel_cols if c not in hand_cols]

p1_df["hand_energy"] = mag(p1_df, hand_cols)
p1_df["body_energy"] = mag(p1_df, body_cols)
p1_df["hand_energy_ratio"] = p1_df["hand_energy"] / (p1_df["body_energy"] + 1e-8)

p2_df["hand_energy"] = mag(p2_df, hand_cols)
p2_df["body_energy"] = mag(p2_df, body_cols)
p2_df["hand_energy_ratio"] = p2_df["hand_energy"] / (p2_df["body_energy"] + 1e-8)

# 5. Rolling std
roll_window = 5
for col in vel_cols + acc_cols:
    roll_name = f"{str(col)}_roll_std"
    p1_df[roll_name] = p1_df.groupby("video_id")[col].transform(
        lambda x: x.rolling(roll_window, min_periods=1).std()
    )
    p2_df[roll_name] = p2_df.groupby("video_id")[col].transform(
        lambda x: x.rolling(roll_window, min_periods=1).std()
    )

# FINAL FEATURE COLS - FORCE AS STRING LIST
exclude = {"video_id", "Frame", "Label", "person_id", "activity_type",
           "knife_sharpness_score", "sharpness_class", "sensor_type", "target"}
feature_cols = [str(c) for c in p1_df.columns if str(c) not in exclude]

print(f"[phase=features] Total features: {len(feature_cols)}")
print("First 5 features:", feature_cols[:5])
print("Last 5 features:", feature_cols[-5:])

[phase=features] Adding 5 new feature groups...


D:\Temp\ipykernel_5972\1032526677.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p1_df[ratio_name] = p1_df[a] / (p1_df[v].abs() + 1e-8)
D:\Temp\ipykernel_5972\1032526677.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p2_df[ratio_name] = p2_df[a] / (p2_df[v].abs() + 1e-8)
D:\Temp\ipykernel_5972\1032526677.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

[phase=features] Total features: 465
First 5 features: ['L5 x_vel', 'L5 y_vel', 'L5 z_vel', 'L3 x_vel', 'L3 y_vel']
Last 5 features: ['Left Foot y_acc_roll_std', 'Left Foot z_acc_roll_std', 'Left Toe x_acc_roll_std', 'Left Toe y_acc_roll_std', 'Left Toe z_acc_roll_std']


D:\Temp\ipykernel_5972\1032526677.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p2_df[roll_name] = p2_df.groupby("video_id")[col].transform(
D:\Temp\ipykernel_5972\1032526677.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p1_df[roll_name] = p1_df.groupby("video_id")[col].transform(
D:\Temp\ipykernel_5972\1032526677.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conc

# Relabel

In [48]:
label_encoder = LabelEncoder().fit(pd.concat([p1_df[target_col], p2_df[target_col]]))
num_classes = len(label_encoder.classes_)

p1_df["target"] = label_encoder.transform(p1_df[target_col])
p2_df["target"] = label_encoder.transform(p2_df[target_col])

print(f"[phase=data] Classes: {label_encoder.classes_}")

[phase=data] Classes: ['blunt' 'medium' 'sharp']


D:\Temp\ipykernel_5972\2924891721.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p1_df["target"] = label_encoder.transform(p1_df[target_col])
D:\Temp\ipykernel_5972\2924891721.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  p2_df["target"] = label_encoder.transform(p2_df[target_col])


# Feature Engineering

In [49]:
print(f"[phase=cv] Starting {n_folds}-fold training")

sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

oof_pred = []
oof_true = []
cv_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(p1_df, p1_df["target"], groups=p1_df["video_id"])):
    print(f"\n=== Fold {fold_idx+1}/{n_folds} ===")
    
    train_df = p1_df.iloc[train_idx].reset_index(drop=True)
    val_df = p1_df.iloc[val_idx].reset_index(drop=True)
    
    # Create windows - now safe because feature_cols are strings
    X_train, y_train = create_windows(train_df, feature_cols, "target", window_size, stride)
    X_val, y_val = create_windows(val_df, feature_cols, "target", window_size, stride)
    
    train_dataset = MotionDataset(X_train, y_train)
    val_dataset = MotionDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    # === Your model, optimizer, training loop here ===
    # model = YourModel(len(feature_cols), num_classes).to(device)
    # ... train ...
    
    # After training, collect OOF
    model.eval()
    fold_preds = []
    fold_targets = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            fold_preds.append(preds)
            fold_targets.append(yb.numpy())
    
    oof_pred.append(np.concatenate(fold_preds))
    oof_true.append(np.concatenate(fold_targets))
    
    cv_fold_metrics.append({
        "fold": fold_idx + 1,
        "train_windows": len(X_train),
        "val_windows": len(X_val),
    })

print("\n[phase=cv] Finished.")

[phase=cv] Starting 10-fold training

=== Fold 1/10 ===


KeyError: 45

In [ ]:
print("[phase=eval] OOF Evaluation")

if oof_true:
    y_true = np.concatenate(oof_true).astype(int)
    y_pred = np.concatenate(oof_pred).astype(int)
    
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
    plt.title("OOF Confusion Matrix")
    plt.tight_layout()
    plt.show()
    
    print("\nClass counts in OOF:")
    for i, name in enumerate(label_encoder.classes_):
        print(f"{name}: {np.sum(y_true == i)}")

int64


# Dynamic Windows by Label Transitions

Detect activity label changes (without groupby), count frames per label, visualize one sample, then build dynamic windows.

In [ ]:
# ====================== SUMMARY ======================
print("\n[phase=eval] CV Fold Metrics Summary")
cv_df = pd.DataFrame(cv_fold_metrics)
print(cv_df.to_string(index=False))

print(f"\nMean Val Accuracy: {cv_df.get('best_val_acc', pd.Series([0])).mean():.4f}")

[phase=runs] Building contiguous runs by video_id + sharpness_class transition
[phase=runs] total_runs=17
[phase=runs] frame count per sharpness_class:
  63: 38338
  64: 25630
  76: 9686
  89: 21467
  90: 13068
[phase=runs] run count per sharpness_class:
  63: 4
  64: 7
  76: 1
  89: 3
  90: 2
[phase=runs] run length stats:
count       17.000000
mean      6364.058824
std       4967.499477
min       1170.000000
50%       4235.000000
max      14621.000000
